In [1]:
import sqlparse
from sqlparse.sql import IdentifierList, Identifier
from sqlparse.tokens import Keyword, DML
import os
import pandas as pd
from collections import defaultdict

from IPython.display import HTML
from IPython.display import display_html

import sys
sys.path.append("..")
from utils.utils import *

import networkx as nx
from networkx.readwrite import json_graph
import pickle

    
def load_sql_rep(fn, dummy=None):
    assert ".pkl" in fn
    try:
        with open(fn, "rb") as f:
            query = pickle.load(f)
    except Exception as e:
        print(e)
        print(fn + " failed to load...")
        exit(-1)

    query["subset_graph"] = nx.OrderedDiGraph(json_graph.adjacency_graph(query["subset_graph"]))
    query["join_graph"] = json_graph.adjacency_graph(query["join_graph"])
    if "subset_graph_paths" in query:
        query["subset_graph_paths"] = nx.OrderedDiGraph(json_graph.adjacency_graph(query["subset_graph_paths"]))

    return query

print("========================================bi_1_ParsingSQLs.py========================================")
WK = "ceb"

folder_path = "./benchmark_small"
# 遍历文件夹
dir_list = os.listdir(folder_path)

for dir_name in dir_list:
    if ".csv" in dir_name:
        continue
    
    QDIR = os.path.join(folder_path, dir_name, "queries_processed")
    OUTPATH = os.path.join(folder_path, dir_name, "dfs")
    make_dir(OUTPATH)

    COLS_TO_TABLES = {}

    qdict = {}
    qfns = []

    QREP_DIR = os.path.join(folder_path, dir_name, "workload")
    if WK in ["job", "ceb"]:
        qfns = os.listdir(QREP_DIR)
        for fn in qfns:
            qfn = os.path.join(QREP_DIR, fn)
            if ".pkl" in qfn:
                qdict[fn] = load_sql_rep(qfn)

    FNS = os.listdir(QDIR)

    sqls = []
    qreps = []

    for fi, fn in enumerate(FNS):
        if ".sql" not in fn:
            continue
        if "all" in fn:
            continue
        
        fnkey = fn.replace(".sql", ".pkl")
        qreps.append(qdict[fnkey])
            
        fn = os.path.join(QDIR, fn)
        with open(fn, "r", encoding='utf-8') as f:
            sql = f.read()

        sql = sql.replace("IN", "in")
        sql = sql.replace("In", "in")
        sql = sql.replace("ILIKE", "like")
        sql = sql.replace("LIKE", "like")
        
        if "return_rank" in sql:
            print(sql)
        sqls.append(sql)
        
    for i, sql in enumerate(sqls):
        sql = sql.replace("IN", "in")
        sql = sql.replace("In", "in")
        sql = sql.replace("ILIKE", "like")
        sql = sql.replace("LIKE", "like")
        sqls[i] = sql

  

========================================bi_1_ParsingSQLs.py========================================


In [2]:
parse_sql_preds(sql)  

,op,dtype,optype,constant,input,column,opstring,is_id
0,=,string,=,'Bogota Sur',"""rentabilidad_1""","""locación""","""rentabilidad_1"".""locación"" = 'Bogota Sur'",0
1,=,string,=,'CE',"""rentabilidad_1""","""zona""","""rentabilidad_1"".""zona"" = 'CE'",0


In [3]:
opdf,exprdf = parse_sqls_par(sqls, COLS_TO_TABLES)

number of sql groups:  1
num_par is  1 

finished parsing batch:  0 in :  0 fails:  0


In [4]:
opdf

,op,dtype,optype,constant,input,column,opstring,is_id,jobid,discrete_ops,cont_ops,like_ops,alias,exprhash
0,=,string,=,'Bogota Sur',"""rentabilidad_1""","""locación""","""rentabilidad_1"".""locación"" = 'Bogota Sur'",0,0,1,0,0,"""rentabilidad_1""",68592321549657620321600124900816096927583765937
1,=,string,=,'CE',"""rentabilidad_1""","""zona""","""rentabilidad_1"".""zona"" = 'CE'",0,0,1,0,0,"""rentabilidad_1""",68592321549657620321600124900816096927583765937
2,=,string,=,'Bogota Sur',"""rentabilidad_1""","""locación""","""rentabilidad_1"".""locación"" = 'Bogota Sur'",0,1,1,0,0,"""rentabilidad_1""",354226151535876892119916199461724716207281837193
3,=,string,=,'CE',"""rentabilidad_1""","""zona""","""rentabilidad_1"".""zona"" = 'CE'",0,1,1,0,0,"""rentabilidad_1""",354226151535876892119916199461724716207281837193
4,=,string,=,'Preventa On Premise',"""rentabilidad_1""","""figura""","""rentabilidad_1"".""figura"" = 'Preventa On Premise'",0,2,1,0,0,"""rentabilidad_1""",1074081901856518500863344009753585848014820234403
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,=,string,=,'OC',"""rentabilidad_1""","""zona""","""rentabilidad_1"".""zona"" = 'OC'",0,23,1,0,0,"""rentabilidad_1""",575563804689980941537408203064381895104946463932
58,=,string,=,'Foraneo',"""rentabilidad_1""","""sede foraneo sintec""","""rentabilidad_1"".""sede foraneo sintec"" = 'Fora...",0,24,1,0,0,"""rentabilidad_1""",1430854988200094793182223858335957073960922870714
59,=,string,=,'NR',"""rentabilidad_1""","""zona""","""rentabilidad_1"".""zona"" = 'NR'",0,24,1,0,0,"""rentabilidad_1""",1430854988200094793182223858335957073960922870714
60,=,string,=,'Bogota Sur',"""rentabilidad_1""","""locación""","""rentabilidad_1"".""locación"" = 'Bogota Sur'",0,25,1,0,0,"""rentabilidad_1""",19910378312979121873993866813656972001427523462


In [5]:
exprdf

,filtersql,jobid,input,alias,exprhash,unique_filter_cols,discrete_ops,cont_ops,like_ops,is_id,num_unique_ops,num_ops,num_discrete_consts,num_likes,like_lens
0,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",0,"""rentabilidad_1""","""rentabilidad_1""",68592321549657620321600124900816096927583765937,2,1,0,0,0,1,2,2,0,0
1,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",1,"""rentabilidad_1""","""rentabilidad_1""",354226151535876892119916199461724716207281837193,2,1,0,0,0,1,2,2,0,0
2,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",2,"""rentabilidad_1""","""rentabilidad_1""",1074081901856518500863344009753585848014820234403,3,1,0,0,0,1,3,3,0,0
3,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",3,"""rentabilidad_1""","""rentabilidad_1""",528740310513859671886471687374341376770706216845,2,1,0,0,0,1,2,2,0,0
4,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",4,"""rentabilidad_1""","""rentabilidad_1""",30391243358831437756649764896058048039357798061,4,1,0,0,0,1,4,4,0,0
5,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",5,"""rentabilidad_1""","""rentabilidad_1""",338094258429485642449962140679804067369069010838,2,1,0,0,0,1,2,2,0,0
6,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",6,"""rentabilidad_1""","""rentabilidad_1""",442034127438150248525596842744800397576838255365,2,1,0,0,0,1,2,2,0,0
7,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",9,"""rentabilidad_1""","""rentabilidad_1""",128711591847766280470884459918906550635784364451,2,1,0,0,0,1,2,2,0,0
8,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",10,"""rentabilidad_1""","""rentabilidad_1""",548116517754048348729066645796794341074909228686,2,1,0,0,0,1,2,2,0,0
9,"SELECT COUNT(*) from ""rentabilidad_1"" AS ""rent...",11,"""rentabilidad_1""","""rentabilidad_1""",370939129191626800623073711729891536802927283236,3,1,0,0,0,1,3,3,0,0
